## TLab IV - Preprocessing
Cleaning the raw dataset: fixing dtypes, handling sentinel values, encoding and bulding trend features.

### C1: Fix the Dtypes

In [8]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/citibike_weather_daily.csv")
df['ride_date'] = pd.to_datetime(df['ride_date'])
df.dtypes

ride_date           datetime64[us]
num_rides                    int64
avg_duration_min           float64
temp_f                     float64
max_temp_f                 float64
min_temp_f                 float64
wind_speed_knots           float64
precip_in                  float64
day_of_week                    str
month                        int64
dtype: object

In [9]:
df['precip_in'] = df['precip_in'].replace(99.99, np.nan)
df['precip_in'].isna().sum()

np.int64(1)

In [10]:
df['precip_in'] = df['precip_in'].fillna(0)
df['precip_in'].isna().sum()

np.int64(0)

### C2: Sentinel Values
- `precip_in` had 1 row coded as 99.99 (NOAA missing-value sentinel)
- Converted to NaN, then imputed as 0
- Justification: precipitation is mostly 0 (dry days dominate); imputing 0 is a conservative default vs. guessing a nonzero amount with no basis. Only 1/1610 rows affected - minimal impact either way

In [11]:
df = pd.get_dummies(df, columns = ['day_of_week'], drop_first=True)
df.head()

,ride_date,num_rides,avg_duration_min,temp_f,max_temp_f,min_temp_f,wind_speed_knots,precip_in,month,day_of_week_Monday,day_of_week_Saturday,day_of_week_Sunday,day_of_week_Thursday,day_of_week_Tuesday,day_of_week_Wednesday
0,2013-07-01,16650,16.309988,74.8,78.1,73.4,7.8,0.00,7,True,False,False,False,False,False
1,2013-07-02,22745,15.968826,76.1,82.9,73.0,8.0,0.73,7,False,False,False,False,True,False
2,2013-07-03,21864,16.238808,78.5,84.9,73.9,8.8,0.06,7,False,False,False,False,False,True
3,2013-07-04,22326,21.218474,82.0,91.0,73.9,8.6,0.96,7,False,False,False,True,False,False
4,2013-07-05,21842,18.040443,84.4,93.0,75.9,9.0,0.00,7,False,False,False,False,False,False


### C3: Encoded Day of Week
- One-hot encoded with 'pd.get_dummies(drop_first=True)'
- Friday dropped as baseline (implied when all 6 dummies are False)
- 'drop_first=True' avoids redundant column (7 days fully determined by 6 flags)

## C4: Build a Trend Feature

In [13]:
df['days_since_launch'] = (df['ride_date'] - df['ride_date'].min()).dt.days
df['year'] = df['ride_date'].dt.year
df[['ride_date', 'days_since_launch', 'year']].head()

,ride_date,days_since_launch,year
0,2013-07-01,0,2013
1,2013-07-02,1,2013
2,2013-07-03,2,2013
3,2013-07-04,3,2013
4,2013-07-05,4,2013


### C4: Trend Feature
- Added 'days_since_launch'
- Captures system growth over time so model knows what year it is
- Will use `days_since_launch` for modeling (continuous, finer_grained than year)

### C6: Save the Clean Dataset

In [15]:
df.to_csv("../data/citibike_weather_daily_clean.csv", index=False)
print("Saved:", df.shape)

Saved: (1610, 17)


### C6: Saved Clean Dataset
- Saved to 'data/citibike_weather_daily_clean.csv'
- Shape: (1610, 17)